# LFM3-TS quickstart

LFM3-TS is a small language model that reads time series: give it a series and a question, and it answers in text.

This notebook loads the model from the Hugging Face Hub, makes a series with a spike in it, and asks where the anomaly is. A GPU runtime is recommended (Runtime > Change runtime type).

In [ ]:
# transformers 5.23 is the first release with LFM3 built in. Colab already has PyTorch: do not upgrade it here.
# If this upgrades transformers, restart the runtime (Runtime > Restart session) before running the next cell.
%pip install -q -U "transformers>=5.23"

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

HUB_ID = "nardi/LFM3-TS"

device = "cuda" if torch.cuda.is_available() else "cpu"
# trust_remote_code: the checkpoint ships its own model code, the files of https://github.com/nardi-xhepi/lfm3-ts
model = AutoModelForCausalLM.from_pretrained(HUB_ID, trust_remote_code=True).to(device).eval()
processor = AutoProcessor.from_pretrained(HUB_ID, trust_remote_code=True)
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion parameters on {device}")

## A series with a spike

A sine wave with a little noise, 512 points long, and one spike at point 300. Points are numbered from 0.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
series = 20 + 5 * np.sin(np.arange(512) / 8) + rng.normal(scale=0.3, size=512)
series[300] += 12

plt.figure(figsize=(10, 3))
plt.plot(series, linewidth=1)
plt.axvline(300, color="tab:orange", alpha=0.4, linewidth=6)
plt.xlabel("point")
plt.show()

## Ask

`model.chat` takes the processor, the series (a 1D array or a list of numbers) and the question, and returns the answer.

In [ ]:
answer = model.chat(processor, series, "Is there an anomaly in this time series? If so, where is it and what does it look like?")
print(answer)

In [ ]:
# other questions on the same series
for question in [
    "Is there an anomaly in this time series? Answer Yes or No first; if Yes, say around which point it is.",
    "What is the period of the fluctuation?",
]:
    print(question)
    print("  ", model.chat(processor, series, question))

## The same series without the spike

The right answer is now "No".

In [ ]:
quiet = series.copy()
quiet[300] -= 12
print(model.chat(processor, quiet, "Is there an anomaly in this time series? Answer Yes or No first; if Yes, say around which point it is."))

## Your own data

Any 1D array works, up to about 2,000 points (the model was trained on series of up to 2,048). NaN marks a missing value.

```python
import pandas as pd
values = pd.read_csv("my_file.csv")["my_column"].to_numpy()
print(model.chat(processor, values[:1024], "Is there an anomaly in this time series? If so, where is it?"))
```